# 溯流 · Workflow Atlas

运行本项目自带的模拟消息，观察 HelloAgents 六阶段编排、原文证据与人工审核边界。使用 Python 3.13 内核，依赖安装步骤见 README。

In [1]:
from pathlib import Path
import os
import sys
import json

ROOT = Path.cwd().resolve()
assert (ROOT / "backend" / "app").is_dir(), "请在项目子目录打开 Notebook"
os.environ["WORKFLOW_ROOT"] = str(ROOT)
sys.path.insert(0, str(ROOT / "backend"))

from app.agents.orchestrator import WorkflowOrchestrator
from app.connectors.local import parse_messages

messages = parse_messages(
    (ROOT / "data" / "fixtures" / "demo_messages.json").read_text(encoding="utf-8"),
    "demo-feishu",
)
print(f"模拟消息数量 {len(messages)}")


模拟消息数量 29


## 六阶段本地分析

通过真实 HelloAgents 工具注册和执行接口运行确定性演示。

In [2]:
result = WorkflowOrchestrator(provider="local-demo").analyze(messages)
assert result.provider == "local-demo"
assert len(result.agent_results) == 6
assert all(stage.status == "completed" for stage in result.agent_results)
proposal = result.proposal
print(json.dumps({
    "provider": result.provider,
    "projects": [project.name for project in proposal.projects],
    "facts": len(proposal.facts),
    "nodes": len(proposal.nodes),
    "edges": len(proposal.edges),
}, ensure_ascii=False, indent=2))


✅ 工具 'Information_Filter' 已注册。
✅ 工具 'Fact_Extractor' 已注册。
✅ 工具 'Project_Resolver' 已注册。
✅ 工具 'Relation_Reasoner' 已注册。
✅ 工具 'Workflow_Reconstructor' 已注册。
✅ 工具 'Evidence_Verifier' 已注册。
{
  "provider": "local-demo",
  "projects": [
    "北辰门户",
    "星河数据平台"
  ],
  "facts": 25,
  "nodes": 24,
  "edges": 31
}


## 回查原文证据

展示前三条事实及其精确引文。引用核验通过后，事实仍需人工审核。

In [3]:
from app.workflow.verifier import verify_evidence

assert proposal.facts
assert all(fact.verified and fact.review_status == "pending" for fact in proposal.facts)
for fact in proposal.facts[:3]:
    assert all(verify_evidence(reference, messages) for reference in fact.evidence)
    print(json.dumps({
        "fact": fact.summary,
        "review_status": fact.review_status,
        "evidence": [reference.model_dump(mode="json") for reference in fact.evidence],
    }, ensure_ascii=False, indent=2))


{
  "fact": "计划周一完成需求梳理",
  "review_status": "pending",
  "evidence": [
    {
      "message_id": "demo-02",
      "source_id": "demo-feishu",
      "quote": "【北辰门户】[任务:需求梳理] 计划周一完成需求梳理"
    },
    {
      "message_id": "demo-02",
      "source_id": "demo-feishu",
      "quote": "【北辰门户】"
    }
  ]
}
{
  "fact": "计划接入订单数据",
  "review_status": "pending",
  "evidence": [
    {
      "message_id": "demo-19",
      "source_id": "demo-feishu",
      "quote": "【星河数据平台】[任务:数据接入] 计划接入订单数据"
    },
    {
      "message_id": "demo-19",
      "source_id": "demo-feishu",
      "quote": "【星河数据平台】"
    }
  ]
}
{
  "fact": "开始接口开发",
  "review_status": "pending",
  "evidence": [
    {
      "message_id": "demo-03",
      "source_id": "demo-feishu",
      "quote": "【北辰门户】[任务:接口开发] 开始接口开发"
    },
    {
      "message_id": "demo-03",
      "source_id": "demo-feishu",
      "quote": "【北辰门户】"
    }
  ]
}


## 临时数据库中的 API 闭环

模拟导入、图谱读取与训练出口核验使用独立临时数据库。

In [4]:
from tempfile import TemporaryDirectory
from fastapi.testclient import TestClient
from app.config import Settings
from app.main import create_app

with TemporaryDirectory(prefix="workflow-atlas-notebook-") as temporary:
    settings = Settings(
        database_path=str(Path(temporary) / "demo.db"),
        provider="local-demo",
        external_llm_enabled=False,
        external_data_authorized=False,
        feishu_sync_enabled=False,
        api_token="",
        _env_file=None,
    )
    with TestClient(create_app(settings)) as client:
        response = client.post("/api/demo")
        assert response.status_code == 200, response.text
        assert response.json()["job"]["status"] == "completed"
        stats = client.get("/api/stats")
        assert stats.status_code == 200
        print(json.dumps(stats.json(), ensure_ascii=False, indent=2))
        training = client.get("/api/export", params={"format": "training"})
        assert training.status_code == 200
        assert not training.text.strip()
        print("全部事实等待人工审核，训练出口为空。")


✅ 工具 'Information_Filter' 已注册。
✅ 工具 'Fact_Extractor' 已注册。
✅ 工具 'Project_Resolver' 已注册。
✅ 工具 'Relation_Reasoner' 已注册。
✅ 工具 'Workflow_Reconstructor' 已注册。
✅ 工具 'Evidence_Verifier' 已注册。
{
  "messages": 29,
  "facts": 25,
  "projects": 2,
  "nodes": 24,
  "edges": 31,
  "pending_review": 25,
  "versions": 1
}
全部事实等待人工审核，训练出口为空。


完整 Web 工作台启动步骤见 README。真实消息的模型分析需在设置页完成连接测试和数据发送授权。